# NMF Parallelization Report - Complete Benchmark Suite

**All-in-one notebook**: Generate matrices → Run benchmarks → Analyze results → Generate figures

## Project Summary

This project compares two NMF algorithms:
- **MU (Multiplicative Update)**: Trivially parallel (Jacobi-style), but slow convergence
- **HALS**: Fast convergence (Gauss-Seidel), but sequential dependencies → non-trivial parallelization

### Key Findings
1. HALS converges ~7x better than MU at same iteration count
2. Block-parallel HALS with random shuffling preserves convergence while enabling parallelism
3. MU optimization limited by Amdahl's Law (cuBLAS dominates 85% of runtime)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import struct
import glob
import os

# Style settings for publication
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 11

# Create output directories
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../data', exist_ok=True)

print('Ready!')

## 1. Generate Low-Rank Test Matrices

Creates matrices `X = W_true @ H_true + noise` with true low-rank structure.
This ensures NMF algorithms can actually converge (unlike random noise).

In [ ]:
def generate_lowrank_matrix(m, n, rank, noise_level=0.01, seed=42):
    """Generate a low-rank non-negative matrix."""
    np.random.seed(seed)
    W_true = np.random.rand(m, rank).astype(np.float32)
    H_true = np.random.rand(rank, n).astype(np.float32)
    X = W_true @ H_true
    X += noise_level * np.random.rand(m, n).astype(np.float32)
    X = np.maximum(X, 0).astype(np.float32)
    return X

def save_matrix_binary(filename, X):
    """Save matrix in binary format for CUDA implementations."""
    m, n = X.shape
    X_col = np.asfortranarray(X)  # Column-major
    with open(filename, 'wb') as f:
        f.write(struct.pack('i', m))
        f.write(struct.pack('i', n))
        f.write(X_col.tobytes())
    print(f'  Saved {m}x{n} matrix to {filename}')

def verify_lowrank(X, rank):
    """Verify the matrix has expected low-rank structure."""
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    energy_ratio = (S[:rank]**2).sum() / (S**2).sum()
    print(f'  Rank-{rank} energy: {energy_ratio*100:.1f}%')
    return energy_ratio

In [ ]:
# Configuration - Full benchmark suite
SIZES = [500, 1000, 2000, 4000, 8000]  # Full range
RANK = 20

print('='*60)
print('Generating Low-Rank Test Matrices')
print('='*60)

for size in SIZES:
    filepath = f'../data/lowrank_{size}.bin'
    if os.path.exists(filepath):
        print(f'{size}x{size} matrix already exists, skipping...')
        continue
    print(f'\n{size}x{size} matrix (rank {RANK}):')
    X = generate_lowrank_matrix(size, size, RANK)
    save_matrix_binary(filepath, X)
    verify_lowrank(X, RANK)

print('\n' + '='*60)
print('Matrix generation complete!')
print('='*60)

## 2. Run Benchmarks

Run all NMF implementations with convergence logging.

In [ ]:
def run_benchmark(cmd, name):
    """Run a benchmark command and capture output."""
    print(f'Running {name}...')
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd='..', timeout=300)
        if result.returncode == 0:
            print(f'  ✓ {name} completed')
            # Extract key metrics from output
            for line in result.stdout.split('\n'):
                if 'Final_Error' in line or 'Time:' in line:
                    print(f'    {line.strip()}')
        else:
            print(f'  ✗ {name} failed: {result.stderr}')
        return result.returncode == 0
    except subprocess.TimeoutExpired:
        print(f'  ✗ {name} timed out')
        return False
    except Exception as e:
        print(f'  ✗ {name} error: {e}')
        return False

In [ ]:
# Run ALL 7 implementations
MU_ITERS = 100
HALS_ITERS = 50
LOG_INTERVAL = 10

# Initialize timing CSV
timing_file = '../results/benchmark_timing.csv'
with open(timing_file, 'w') as f:
    f.write('size,method,time_ms,iterations,time_per_iter_ms,final_error\n')

def extract_metrics(output):
    """Extract time and error from program output."""
    import re
    time_match = re.search(r'Time: ([\d.]+)', output)
    error_match = re.search(r'Final_Error: ([\d.e+-]+)', output)
    time = float(time_match.group(1)) if time_match else 0
    error = float(error_match.group(1)) if error_match else 0
    return time, error

print('='*60)
print('Running ALL 7 Implementations')
print('='*60)

for size in SIZES:
    print(f'\n{"="*50}')
    print(f'Matrix Size: {size}x{size}')
    print(f'{"="*50}')
    
    data_file = f'data/lowrank_{size}.bin'
    
    # 1. MU Naive (v1) - baseline
    result = subprocess.run(f'./nmf_naive {data_file} {RANK} {MU_ITERS}', 
                           shell=True, capture_output=True, text=True, cwd='..')
    time, error = extract_metrics(result.stdout)
    if time > 0:
        with open(timing_file, 'a') as f:
            f.write(f'{size},mu_naive,{time},{MU_ITERS},{time/MU_ITERS:.4f},{error}\n')
        print(f'MU Naive: {time:.1f} ms, Error: {error:.4e}')
    
    # 2. MU L2 (Memory-Opt)
    result = subprocess.run(f'./nmf_memory_opt {data_file} {RANK} {MU_ITERS}', 
                           shell=True, capture_output=True, text=True, cwd='..')
    time, error = extract_metrics(result.stdout)
    if time > 0:
        with open(timing_file, 'a') as f:
            f.write(f'{size},mu_l2_memory,{time},{MU_ITERS},{time/MU_ITERS:.4f},{error}\n')
        print(f'MU L2 Memory: {time:.1f} ms, Error: {error:.4e}')
    
    # 3. MU L3 (Compute-Opt)
    result = subprocess.run(f'./nmf_compute_opt {data_file} {RANK} {MU_ITERS} 128', 
                           shell=True, capture_output=True, text=True, cwd='..')
    time, error = extract_metrics(result.stdout)
    if time > 0:
        with open(timing_file, 'a') as f:
            f.write(f'{size},mu_l3_compute,{time},{MU_ITERS},{time/MU_ITERS:.4f},{error}\n')
        print(f'MU L3 Compute: {time:.1f} ms, Error: {error:.4e}')
    
    # 4. HALS CPU
    result = subprocess.run(f'./nmf_hals_cpu {data_file} {RANK} {HALS_ITERS}', 
                           shell=True, capture_output=True, text=True, cwd='..', timeout=600)
    time, error = extract_metrics(result.stdout)
    if time > 0:
        with open(timing_file, 'a') as f:
            f.write(f'{size},hals_cpu,{time},{HALS_ITERS},{time/HALS_ITERS:.4f},{error}\n')
        print(f'HALS CPU: {time:.1f} ms, Error: {error:.4e}')
    
    # 5. HALS GPU Strict
    result = subprocess.run(f'./nmf_hals_gpu_strict {data_file} {RANK} {HALS_ITERS}', 
                           shell=True, capture_output=True, text=True, cwd='..')
    time, error = extract_metrics(result.stdout)
    if time > 0:
        with open(timing_file, 'a') as f:
            f.write(f'{size},hals_gpu_strict,{time},{HALS_ITERS},{time/HALS_ITERS:.4f},{error}\n')
        print(f'HALS GPU Strict: {time:.1f} ms, Error: {error:.4e}')
    
    # 6. HALS GPU Block
    result = subprocess.run(f'./nmf_hals_gpu_block {data_file} {RANK} {HALS_ITERS} 5', 
                           shell=True, capture_output=True, text=True, cwd='..')
    time, error = extract_metrics(result.stdout)
    if time > 0:
        with open(timing_file, 'a') as f:
            f.write(f'{size},hals_gpu_block,{time},{HALS_ITERS},{time/HALS_ITERS:.4f},{error}\n')
        print(f'HALS GPU Block: {time:.1f} ms, Error: {error:.4e}')

print('\n' + '='*60)
print('All benchmarks complete!')
print(f'Results saved to: {timing_file}')
print('='*60)

In [ ]:
# Multi-GPU Benchmarks (run on cluster with 2+ GPUs)
# Uncomment when running on cluster

# print('='*60)
# print('Running Multi-GPU Benchmarks')
# print('='*60)

# for size in [1000, 2000]:
#     run_benchmark(
#         f'./nmf_multigpu data/lowrank_{size}.bin {RANK} {MU_ITERS} 2 --log-convergence {LOG_INTERVAL}',
#         f'MU L4 2-GPU ({size})'
#     )
#     os.rename('../results/convergence_mu_l4.csv', f'../results/convergence_mu_l4_{size}_2gpu.csv')

print('Multi-GPU tests: Uncomment cell above when running on cluster')

## 3. Load Results

In [ ]:
results_dir = '../results'

def load_convergence(pattern):
    """Load convergence CSV files matching pattern."""
    files = glob.glob(os.path.join(results_dir, pattern))
    data = {}
    for f in files:
        name = os.path.basename(f).replace('.csv', '')
        try:
            df = pd.read_csv(f)
            data[name] = df
            print(f'Loaded {name}: {len(df)} rows')
        except Exception as e:
            print(f'Error loading {f}: {e}')
    return data

convergence_data = load_convergence('convergence_*.csv')
print(f'\nLoaded {len(convergence_data)} convergence files')

## 4. Convergence Plot: MU vs HALS

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'mu_l2': '#1f77b4', 'mu_l3': '#2ca02c', 'mu_l4': '#9467bd', 'hals': '#d62728'}

for name, df in convergence_data.items():
    if 'iteration' in df.columns and 'error' in df.columns:
        # Determine color and label
        if 'mu_l2' in name:
            color, label = colors['mu_l2'], 'MU L2 (Memory-Opt)'
        elif 'mu_l3' in name:
            color, label = colors['mu_l3'], 'MU L3 (Compute-Opt)'
        elif 'mu_l4' in name:
            color, label = colors['mu_l4'], 'MU L4 (Multi-GPU)'
        else:
            color, label = colors['hals'], 'HALS Block-Parallel'
        
        # Extract size if present
        for part in name.split('_'):
            if part.isdigit():
                label += f' ({part})'
                break
        
        ax.plot(df['iteration'], df['error'], label=label, color=color, linewidth=2, alpha=0.8)

ax.set_xlabel('Iteration')
ax.set_ylabel('Relative Error')
ax.set_title('Convergence Comparison: MU vs HALS')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/convergence_comparison.png', dpi=150, bbox_inches='tight')
plt.savefig('../results/figures/convergence_comparison.pdf', bbox_inches='tight')
plt.show()

print('Saved: results/figures/convergence_comparison.png')

## 5. Scaling Plot: Time vs Matrix Size

In [ ]:
def extract_metrics(convergence_data):
    """Extract final metrics for each method/size."""
    metrics = []
    for name, df in convergence_data.items():
        # Extract size
        size = 1000
        for p in name.split('_'):
            if p.isdigit():
                size = int(p)
                break
        
        # Get method
        if 'mu_l2' in name:
            method = 'MU L2'
        elif 'mu_l3' in name:
            method = 'MU L3'
        elif 'mu_l4' in name:
            method = 'MU L4'
        elif 'hals' in name:
            method = 'HALS'
        else:
            method = name
        
        total_time = df['time_ms'].sum() if 'time_ms' in df.columns else 0
        final_error = df['error'].iloc[-1]
        
        metrics.append({
            'method': method,
            'size': size,
            'total_time_ms': total_time,
            'final_error': final_error,
            'iterations': len(df)
        })
    return pd.DataFrame(metrics)

metrics_df = extract_metrics(convergence_data)
print(metrics_df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors_list = ['#1f77b4', '#2ca02c', '#9467bd', '#d62728']
markers = ['o', 's', '^', 'D']

for i, method in enumerate(metrics_df['method'].unique()):
    method_data = metrics_df[metrics_df['method'] == method].sort_values('size')
    if len(method_data) > 0:
        ax.plot(method_data['size'], method_data['total_time_ms'], 
                marker=markers[i % len(markers)], 
                color=colors_list[i % len(colors_list)],
                label=method, linewidth=2, markersize=8)

ax.set_xlabel('Matrix Size (n x n)')
ax.set_ylabel('Total Time (ms)')
ax.set_title('Scaling: Time vs Matrix Size')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/scaling_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print('Saved: results/figures/scaling_plot.png')

## 5b. Load Full Benchmark Results

Load results from `benchmark_timing.csv` (from shell script or cell above).

In [ ]:
# Load benchmark timing data
timing_file = '../results/benchmark_timing.csv'

if os.path.exists(timing_file):
    timing_df = pd.read_csv(timing_file)
    print(f'Loaded {len(timing_df)} benchmark results\n')
    print(timing_df.to_string(index=False))
    
    # Create comprehensive scaling plot with all methods
    fig, ax = plt.subplots(figsize=(12, 7))
    
    method_colors = {
        'mu_naive': '#1f77b4',
        'mu_l2_memory': '#2ca02c',
        'mu_l3_compute': '#9467bd',
        'hals_cpu': '#ff7f0e',
        'hals_gpu_strict': '#d62728',
        'hals_gpu_block': '#8c564b'
    }
    
    method_labels = {
        'mu_naive': 'MU Naive (baseline)',
        'mu_l2_memory': 'MU L2 (Memory-Opt)',
        'mu_l3_compute': 'MU L3 (Compute-Opt)',
        'hals_cpu': 'HALS CPU',
        'hals_gpu_strict': 'HALS GPU Strict',
        'hals_gpu_block': 'HALS GPU Block'
    }
    
    markers = ['o', 's', '^', 'D', 'v', 'p']
    
    for i, method in enumerate(timing_df['method'].unique()):
        method_data = timing_df[timing_df['method'] == method].sort_values('size')
        color = method_colors.get(method, 'gray')
        label = method_labels.get(method, method)
        ax.plot(method_data['size'], method_data['time_ms'], 
                marker=markers[i % len(markers)], 
                color=color,
                label=label, linewidth=2, markersize=8)
    
    ax.set_xlabel('Matrix Size (n x n)')
    ax.set_ylabel('Total Time (ms)')
    ax.set_title('Scaling: All 7 NMF Implementations')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')  # Log scale for better visibility
    
    plt.tight_layout()
    plt.savefig('../results/figures/scaling_all_methods.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/figures/scaling_all_methods.png')
else:
    print(f'No timing file found at {timing_file}')
    print('Run the benchmark cell above first.')

## 5c. Roofline Analysis

Calculate theoretical FLOPS and memory movement, plot roofline model.

In [ ]:
# Roofline Analysis
# GPU specs (RTX 3050)
PEAK_GFLOPS = 9100  # FP32
PEAK_BANDWIDTH = 224  # GB/s

def compute_nmf_flops(m, n, k):
    """Compute theoretical FLOPs for one NMF iteration."""
    # GEMM operations dominate
    flops = 2*k*k*m + 2*k*n*m + 2*k*n*k  # H update
    flops += 2*k*k*n + 2*m*k*n + 2*m*k*k  # W update
    flops += 4*(k*n + m*k)  # Element-wise
    return flops

def compute_nmf_bytes(m, n, k):
    """Compute memory movement for one NMF iteration (bytes)."""
    # Read/write matrices
    bytes_moved = m*n*4 + m*k*4*2 + k*n*4*2  # X, W, H
    bytes_moved += (k*k + k*n + m*k)*4  # Intermediates
    return bytes_moved

# Calculate for all sizes
roofline_data = []
for size in [500, 1000, 2000, 4000, 8000]:
    flops = compute_nmf_flops(size, size, RANK)
    bytes_moved = compute_nmf_bytes(size, size, RANK)
    ai = flops / bytes_moved
    roofline_data.append({
        'size': size,
        'flops_per_iter': flops,
        'bytes_per_iter': bytes_moved,
        'arithmetic_intensity': ai,
        'theoretical_peak': min(ai * PEAK_BANDWIDTH, PEAK_GFLOPS)
    })

roofline_df = pd.DataFrame(roofline_data)
print('Theoretical Roofline Data:')
print(roofline_df.to_string(index=False))

# Plot roofline model
fig, ax = plt.subplots(figsize=(12, 8))

# Roofline ceiling
ai_range = np.logspace(-1, 3, 100)
memory_bound = ai_range * PEAK_BANDWIDTH
compute_bound = np.full_like(ai_range, PEAK_GFLOPS)
roofline = np.minimum(memory_bound, compute_bound)

ax.loglog(ai_range, roofline, 'k-', linewidth=3, label='Roofline')
ax.axhline(y=PEAK_GFLOPS, color='gray', linestyle='--', alpha=0.5, 
           label=f'Peak Compute ({PEAK_GFLOPS} GFLOPS)')

# Ridge point
ridge_ai = PEAK_GFLOPS / PEAK_BANDWIDTH
ax.axvline(x=ridge_ai, color='red', linestyle=':', alpha=0.7)
ax.annotate(f'Ridge Point\nAI={ridge_ai:.1f}', xy=(ridge_ai*1.1, PEAK_GFLOPS/3), fontsize=10)

# Plot achieved performance from timing data
if 'timing_df' in dir() and timing_df is not None:
    colors = plt.cm.tab10(np.linspace(0, 1, 7))
    
    for i, (_, row) in enumerate(timing_df.iterrows()):
        size = row['size']
        time_ms = row['time_ms']
        iterations = row['iterations']
        method = row['method']
        
        flops = compute_nmf_flops(size, size, RANK) * iterations
        bytes_moved = compute_nmf_bytes(size, size, RANK) * iterations
        ai = flops / bytes_moved
        achieved_gflops = flops / (time_ms * 1e6)  # Convert ms to seconds
        
        ax.scatter(ai, achieved_gflops, s=100, alpha=0.7,
                  label=f'{method} ({size})' if size == 1000 else '')

ax.set_xlabel('Arithmetic Intensity (FLOPs/Byte)', fontsize=14)
ax.set_ylabel('Performance (GFLOPS)', fontsize=14)
ax.set_title('Roofline Model - RTX 3050', fontsize=16)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 1000)
ax.set_ylim(0.1, PEAK_GFLOPS * 2)
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../results/figures/roofline_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/roofline_plot.png')

## 5d. Speedup Analysis

Calculate speedup relative to baselines (MU Naive and HALS CPU).

In [ ]:
# Calculate speedup relative to baselines
if 'timing_df' in dir() and timing_df is not None:
    speedup_data = []
    
    for size in timing_df['size'].unique():
        size_data = timing_df[timing_df['size'] == size]
        
        # Get baselines
        mu_naive_time = size_data[size_data['method'] == 'mu_naive']['time_ms'].values
        hals_cpu_time = size_data[size_data['method'] == 'hals_cpu']['time_ms'].values
        
        mu_baseline = mu_naive_time[0] if len(mu_naive_time) > 0 else None
        hals_baseline = hals_cpu_time[0] if len(hals_cpu_time) > 0 else None
        
        for _, row in size_data.iterrows():
            speedup_vs_mu = mu_baseline / row['time_ms'] if mu_baseline else None
            speedup_vs_hals = hals_baseline / row['time_ms'] if hals_baseline else None
            
            speedup_data.append({
                'size': size,
                'method': row['method'],
                'time_ms': row['time_ms'],
                'speedup_vs_mu_naive': speedup_vs_mu,
                'speedup_vs_hals_cpu': speedup_vs_hals
            })
    
    speedup_df = pd.DataFrame(speedup_data)
    
    print('='*80)
    print('SPEEDUP ANALYSIS')
    print('='*80)
    print('\n### Speedup vs MU Naive Baseline')
    mu_pivot = speedup_df.pivot(index='method', columns='size', values='speedup_vs_mu_naive')
    print(mu_pivot.round(2).to_string())
    
    print('\n### Speedup vs HALS CPU Baseline')
    hals_pivot = speedup_df.pivot(index='method', columns='size', values='speedup_vs_hals_cpu')
    print(hals_pivot.round(2).to_string())
    
    # Plot speedup bar chart
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # MU speedup
    ax = axes[0]
    mu_methods = ['mu_l2_memory', 'mu_l3_compute']
    x = np.arange(len(timing_df['size'].unique()))
    width = 0.35
    
    for i, method in enumerate(mu_methods):
        method_data = speedup_df[speedup_df['method'] == method].sort_values('size')
        ax.bar(x + i*width, method_data['speedup_vs_mu_naive'], width, 
               label=method.replace('_', ' ').title())
    
    ax.set_xlabel('Matrix Size')
    ax.set_ylabel('Speedup vs MU Naive')
    ax.set_title('MU Optimization Speedup')
    ax.set_xticks(x + width/2)
    ax.set_xticklabels(sorted(timing_df['size'].unique()))
    ax.legend()
    ax.axhline(y=1, color='k', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.3, axis='y')
    
    # HALS speedup
    ax = axes[1]
    hals_methods = ['hals_gpu_strict', 'hals_gpu_block']
    
    for i, method in enumerate(hals_methods):
        method_data = speedup_df[speedup_df['method'] == method].sort_values('size')
        ax.bar(x + i*width, method_data['speedup_vs_hals_cpu'], width,
               label=method.replace('_', ' ').title())
    
    ax.set_xlabel('Matrix Size')
    ax.set_ylabel('Speedup vs HALS CPU')
    ax.set_title('HALS GPU Speedup')
    ax.set_xticks(x + width/2)
    ax.set_xticklabels(sorted(timing_df['size'].unique()))
    ax.legend()
    ax.axhline(y=1, color='k', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('../results/figures/speedup_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: results/figures/speedup_comparison.png')
else:
    print('No timing data available. Run benchmarks first.')

## 6. Multi-GPU Communication Breakdown

In [ ]:
multigpu_files = glob.glob(os.path.join(results_dir, 'convergence_mu_l4*.csv'))

if multigpu_files:
    print('Multi-GPU files found:')
    for f in multigpu_files:
        df = pd.read_csv(f)
        print(f'\n{os.path.basename(f)}:')
        
        if 'compute_ms' in df.columns and 'comm_ms' in df.columns:
            total_compute = df['compute_ms'].iloc[-1]
            total_comm = df['comm_ms'].iloc[-1]
            total = total_compute + total_comm
            
            print(f'  Compute: {total_compute:.2f} ms ({100*total_compute/total:.1f}%)')
            print(f'  Communication: {total_comm:.2f} ms ({100*total_comm/total:.1f}%)')
            
            # Pie chart
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.pie([total_compute, total_comm], 
                   explode=(0, 0.05),
                   labels=[f'Compute\n{total_compute:.1f}ms', f'Communication\n{total_comm:.1f}ms'],
                   colors=['#2ecc71', '#e74c3c'],
                   autopct='%1.1f%%', startangle=90, textprops={'fontsize': 14})
            ax.set_title('Multi-GPU Time Breakdown', fontsize=16)
            
            plt.tight_layout()
            plt.savefig('../results/figures/multigpu_breakdown.png', dpi=150, bbox_inches='tight')
            plt.show()
else:
    print('No multi-GPU data yet.')
    print('Run on cluster: ./nmf_multigpu data/lowrank_1000.bin 20 100 2 --log-convergence 10')

## 7. Summary Table for Report

In [ ]:
print('='*70)
print('SUMMARY TABLE FOR REPORT')
print('='*70)

print('\n### Convergence Comparison')
print('| Method | Size | Iterations | Final Error | Total Time (ms) |')
print('|--------|------|------------|-------------|-----------------|')

for _, row in metrics_df.sort_values(['size', 'method']).iterrows():
    print(f"| {row['method']} | {row['size']} | {row['iterations']} | {row['final_error']:.4e} | {row['total_time_ms']:.1f} |")

print('\n### Key Insights')
print('1. HALS converges ~7x better than MU at same iteration count')
print('2. Block-parallel HALS preserves Gauss-Seidel convergence')
print('3. MU optimization limited by Amdahl\'s Law (cuBLAS dominates)')

In [ ]:
# Export summary markdown
with open('../results/figures/summary.md', 'w') as f:
    f.write('# NMF Benchmark Summary\n\n')
    f.write('## Results\n\n')
    f.write('| Method | Size | Final Error | Time (ms) |\n')
    f.write('|--------|------|-------------|-----------|\n')
    for _, row in metrics_df.sort_values(['size', 'method']).iterrows():
        f.write(f"| {row['method']} | {row['size']} | {row['final_error']:.4e} | {row['total_time_ms']:.1f} |\n")
    f.write('\n## Figures\n\n')
    f.write('- convergence_comparison.png\n')
    f.write('- scaling_plot.png\n')
    f.write('- multigpu_breakdown.png\n')

print('Saved: results/figures/summary.md')

## Done!

### Output Files
- `results/figures/convergence_comparison.png` - MU vs HALS convergence
- `results/figures/scaling_plot.png` - Time vs matrix size
- `results/figures/multigpu_breakdown.png` - Communication overhead (multi-GPU)
- `results/figures/summary.md` - Markdown summary

### For Cluster (Multi-GPU)
Uncomment the multi-GPU cell and run on a machine with 2+ GPUs.